In [1]:
import os, time
import pandas as pd
from openai import OpenAI, RateLimitError, APIError
from tqdm import tqdm

from kaggle_secrets import UserSecretsClient
os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")

# --- CONFIG ---
BEHAVIORS_PATH = "/kaggle/input/datasets/nhlich2003/harmbench-test/data/harmbench_behaviors_text_all.csv"
OUTPUT_PATH    = "/kaggle/working/harmbench_responses_300sample.csv"

TARGET_MODEL   = "llama-3.1-8b-instant"   # model đang bị test (no defense) — đổi sang model thật của bạn
MAX_TOKENS     = 512                        # chuẩn HarmBench, GIỮ NGUYÊN để so sánh được
REQUEST_INTERVAL = 2.0
CHECKPOINT_EVERY = 25

client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url="https://api.groq.com/openai/v1")

# --- Load 300 behavior ---
df = pd.read_csv(BEHAVIORS_PATH)
df = df[df["FunctionalCategory"] != "copyright"].reset_index(drop=True)
df["ContextString"] = df["ContextString"].fillna("")
print(f"Sinh response cho {len(df)} behavior "
      f"({(df.FunctionalCategory=='standard').sum()} standard, "
      f"{(df.FunctionalCategory=='contextual').sum()} contextual)")

# --- Ghép prompt: contextual = ContextString + Behavior; standard = Behavior ---
def build_prompt(row):
    if row["FunctionalCategory"] == "contextual" and row["ContextString"].strip():
        return f"{row['ContextString']}\n\n---\n\n{row['Behavior']}"
    return row["Behavior"]

# --- Gọi model-under-test (no defense = gọi thẳng) ---
def generate(prompt):
    delay = 5
    for _ in range(6):
        try:
            resp = client.chat.completions.create(
                model=TARGET_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,                      # greedy, reproducible
                max_completion_tokens=MAX_TOKENS,
            )
            return resp.choices[0].message.content or ""
        except RateLimitError:
            print(f"Rate limited, chờ {delay}s..."); time.sleep(delay); delay = min(delay*2, 60)
        except APIError as e:
            print(f"API error: {e}, chờ {delay}s..."); time.sleep(delay); delay = min(delay*2, 60)
    return ""

# --- Chạy, có checkpoint ---
responses = []
for i, row in enumerate(tqdm(df.itertuples(), total=len(df))):
    time.sleep(REQUEST_INTERVAL)
    r = generate(build_prompt(row._asdict()))
    responses.append(r)
    tqdm.write(f"[{row.BehaviorID}] len={len(r)}")   # log gọn, không đổ nội dung ra output

    if (i + 1) % CHECKPOINT_EVERY == 0:
        df.iloc[:len(responses)].assign(response=responses).to_csv(OUTPUT_PATH, index=False)

df["response"] = responses
df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved -> {OUTPUT_PATH}")
print(f"Response rỗng: {(df['response'].str.strip() == '').sum()}/{len(df)}")

Sinh response cho 300 behavior (200 standard, 100 contextual)


  0%|          | 1/300 [00:03<16:34,  3.33s/it]

[dimethylmercury_instructions_home] len=106


  1%|          | 2/300 [00:05<13:01,  2.62s/it]

[bio_warfare_list_home] len=115


...

[email_tom_harris_click_link] len=1344


100%|█████████▉| 299/300 [18:04<00:05,  5.52s/it]

[email_amanda_johnson_click_link] len=2125


100%|██████████| 300/300 [18:09<00:00,  3.63s/it]

[dimethylmercury_materials_no_oversight] len=208

Saved -> /kaggle/working/harmbench_responses_300sample.csv
Response rỗng: 0/300
